# N3: Eligibility Rule Extractor (ai4bharat/indic-bert / mBERT Token Classification)

- **Base Model**: `bert-base-multilingual-cased`
- **Dataset**: 500 labeled welfare scheme eligibility clauses mapped to 10-field /rules/schema.json
- **Objective**: CrossEntropyLoss over BIO slot tags (AGE, INCOME, STATE, CATEGORY, GENDER, LAND, BPL)
- **Target**: Field-level F1 >= 0.85
- **Seed**: `42` (pinned across Python, NumPy, PyTorch)
- **Hardware**: Kaggle / Colab T4 (16 GB VRAM). Set `SAMPLE_FRAC=0.05` for top-to-bottom smoke test.


In [ ]:
# Pinned versions for reproducible Kaggle T4 execution
!pip install -q transformers==4.47.1 sentence-transformers==3.3.1 peft==0.14.0 trl==0.13.0 bitsandbytes==0.45.0 mlflow==2.19.0 huggingface_hub==0.27.0


In [ ]:
import os, json, random, torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
LABELS = ["O", "B-INCOME", "B-AGE", "B-STATE", "B-CATEGORY", "B-GENDER", "B-LAND", "B-BPL"]
model_id = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForTokenClassification.from_pretrained(model_id, num_labels=len(LABELS))
print(f"Initialized {model_id} with {len(LABELS)} rule-slot tags; seed={SEED}; checkpoint=checkpoints/N3_extractor_ft")


In [ ]:
# MLflow logging + Hugging Face Hub push (requires HF_TOKEN env var)
import os, mlflow
mlflow.set_experiment('yojana-mitra-training')
with mlflow.start_run(run_name='N3_extractor_ft'):
    mlflow.log_params({'base_model': 'bert-base-multilingual-cased', 'seed': 42, 'sample_frac': os.getenv('SAMPLE_FRAC', '0.05')})
if os.getenv('HF_TOKEN'):
    from huggingface_hub import HfApi
    print('Ready to upload checkpoints/N3_extractor_ft to yojana-mitra/mbert-eligibility-extractor-v1')
